# FuXi Checkpoint Comparison: emb_768 vs emb_1536

This notebook compares two pretrained FuXi checkpoints on the same test split using **latitude-weighted RMSE** and **ACC**.

Checkpoints compared:
- `emb_768`: `/home/raj.ayush/fuxi-final/fuxi_new/Models_paper/pretrain/emb_768/best.pt`
- `emb_1536`: `/home/raj.ayush/fuxi-final/fuxi_new/Models_paper/pretrain/emb_1536/best.pt`

What this notebook provides:
- Fair, side-by-side evaluation on the same samples
- Per-variable and per-lead-time RMSE/ACC comparison
- "Which model is better" analysis with win counts and deltas
- Publication-style plots and CSV outputs for deeper study

Notes:
- Designed to run in conda env `weather_forecast`
- Uses GPU if available
- Tries to use your climatology store first; if inaccessible, falls back to train-period climatology from the main ERA5 store

## Metric Definitions

We follow the latitude-weighted RMSE and ACC definitions you shared.

For variable $c$ at lead time $\tau$:

$$
\mathrm{RMSE}(c,\tau)=\frac{1}{|D|}\sum_{t_0\in D}\sqrt{\frac{1}{H\times W}\sum_{i=1}^{H}\sum_{j=1}^{W}a_i\left(\hat{X}_{c,i,j}^{t_0+\tau}-X_{c,i,j}^{t_0+\tau}\right)^2}
$$

$$
\mathrm{ACC}(c,\tau)=\frac{1}{|D|}\sum_{t_0\in D}
\frac{\sum_{i,j}a_i\left(\hat{X}_{c,i,j}^{t_0+\tau}-M_{c,i,j}^{t_0+\tau}\right)\left(X_{c,i,j}^{t_0+\tau}-M_{c,i,j}^{t_0+\tau}\right)}
{\sqrt{\left(\sum_{i,j}a_i\left(\hat{X}_{c,i,j}^{t_0+\tau}-M_{c,i,j}^{t_0+\tau}\right)^2\right)
\left(\sum_{i,j}a_i\left(X_{c,i,j}^{t_0+\tau}-M_{c,i,j}^{t_0+\tau}\right)^2\right)}}
$$

Where:
- $a_i = \cos(\text{latitude}_i)$ is latitude-area weighting.
- $M$ is climatology.
- $D$ is the set of initialization times.
- Grid is $H \times W$.

In [ ]:
from __future__ import annotations

import gc
import json
import math
import warnings
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch.utils.data import DataLoader, Dataset
import xarray as xr
import zarr

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 10